In [1]:
# Add on top of each image of the dataset a random debris image from the NonPollen image class.

In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import CLASSIFICATION_DATA


In [7]:
import os
import random
import numpy as np
from PIL import Image

In [ ]:

# path dataset image -  original
p_dataset = str(CLASSIFICATION_DATA / "1.IMAGES_classification/")


p_debris = str(CLASSIFICATION_DATA / "1.IMAGES_classification/NonPollen/")

p_dataset_augmented = str(CLASSIFICATION_DATA / "IMAGES_augmented/")


In [35]:
li_debris_p = list([p_debris + "/" + el for el in list(os.listdir(p_debris))])
print(len(li_debris_p))  # for a total of 1827 images,


1827


In [36]:
def paste_debris_on_pollen(clean, debris):
    """
    Take a numpy image of target img (ex. pollen) and of a debris randomly picked among "NonPollen" category
    clean: np.array of the clean pollen image (i.e. without the debris added on top of it), of format [H, W, C]
    debris: np.array of a debris image (to be added on top of the clean pollen image), of format [h, w, C]
    Paste elliptical (circular) patches from debris image onto clean pollen image.
    """
    h_clean, w_clean = clean.shape[:2]
    h_debris, w_debris = debris.shape[:2]

    yy, xx = np.ogrid[:h_debris, :w_debris]
    cy, cx = h_debris // 2, w_debris // 2
    radius = min(h_debris, w_debris) // 2
    mask = (yy - cy)**2 + (xx - cx)**2 <= radius**2
    alpha_mask = np.zeros((h_debris, w_debris), dtype=np.float32)
    

    # Random position, allowing partial out-of-frame of the debris when added on top of the image
    clean_area =  h_clean * w_clean
    debris_area =  w_debris*h_debris
    x = random.randint(-w_debris // 2, w_clean - w_debris // 2)
    y = random.randint(-h_debris // 2, h_clean - h_debris // 2)
    seuil=0.4*clean_area # threshold to determine the alpha of the debris, to avoid covering more than 40% of the pollen image with non-transparent debris
    if debris_area <seuil:
        alpha_mask[mask] = random.uniform(0.6, 0.9) # low transparence level for debris covering small protion of the pollen image
    else:
        alpha_mask[mask] = random.uniform(0.2, 0.6) # high transparence level for debris largely covering the pollen image

    # Clip bounds to image size
    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + w_debris, w_clean)
    y2 = min(y + h_debris, h_clean)

    # Corresponding debris crop
    dx1 = x1 - x
    dy1 = y1 - y
    dx2 = dx1 + (x2 - x1)
    dy2 = dy1 + (y2 - y1)

    debris_crop = debris[dy1:dy2, dx1:dx2]
    alpha_crop = alpha_mask[dy1:dy2, dx1:dx2]

    # Blend only visible region
    for c in range(3):
        clean[y1:y2, x1:x2, c] = (
            clean[y1:y2, x1:x2, c] * (1 - alpha_crop) +
            debris_crop[:, :, c] * alpha_crop)

    return clean.astype(np.uint8)

In [38]:

def create_augmented_dataset(p_dataset, p_dataset_augmented, li_debris_p):
    os.makedirs(p_dataset_augmented, exist_ok=True)
    for cli in sorted(list(os.listdir(p_dataset))):
        print(cli)
        path_class = p_dataset + "/" + cli + "/"
        files = sorted(list(os.listdir(path_class)))
        for file in files:
            if file.lower().endswith('.jpg'):
                img_directory_out = p_dataset_augmented  + "/" + cli + "/"
                os.makedirs(img_directory_out, exist_ok=True)
                p_debris = random.choice(li_debris_p)
                clean = Image.open(path_class+file).convert('L')
                clean0 = clean.size
                debris = Image.open(p_debris).convert('L')
                clean_np = np.stack([np.array(clean)]*3, axis=-1)
                debris_np = np.stack([np.array(debris)]*3, axis=-1)
                augmented = paste_debris_on_pollen(clean_np, debris_np)
                augmented = Image.fromarray(augmented)
                clean1 = augmented.size
                if clean0!=clean1:
                    print("error")
                augmented.save(img_directory_out  +file.replace(".jpg", "_augm.jpg"))
                clean.close()
                debris.close()


create_augmented_dataset(p_dataset, p_dataset_augmented, li_debris_p)              

Acacia
Acer
Alnus
Amarantaceae
Artemisia
Betulaceae
Brassicaceae
Buxus
Carduus
Caryophyllaceae
Cichorioideae
Cupressaceae
Cyperaceae
Echium
Ericaceae
Fagus
FraxinusExcelsior
FraxinusOrnus
Gallium
IndetBlurry
IndetCovered
Juglans
Lamiaceae
Liliaceae
Lycopodium
Moraceae
Morphotype1_smallgrains
Morphotype2_largegrains
Myrtaceae
NonPollen
Olea
Other
Phillyrea
Pinaceae
Pistacia
Plantago
Platanus
Poaceae
PopulusSp
QuercusDeciduous
QuercusIlex
Ranunculaceae
Rhamnus
Rosaceae
Rumex
Salix
Sanguisorba
Tilia
Ulmus
Urtica
ViburnumSambucusTp
VitisF
VitisS
XanthiumAmbrosia
